# NB-01: Citation & Bibliography Audit

Audits every `\cite` call against `\bibitem` keys. Pre-identified issues: `koza1994genetic` missing; two pairs of duplicate bibitems.

In [1]:
import collections
import re
from pathlib import Path

TEX_FILE = "jmlr-hypatiax-paper-final.tex"  # place .tex beside this notebook
tex_path = Path(TEX_FILE)
assert tex_path.exists(), f"File not found: {tex_path.resolve()}"
source = tex_path.read_text(encoding="utf-8")
lines  = source.splitlines()
print(f"Loaded: {tex_path}  |  {len(source):,} chars  |  {len(lines)} lines")

Loaded: jmlr-hypatiax-paper-final.tex  |  114,596 chars  |  2180 lines


## Step 1 — Extract cited keys

In [2]:
CITE_RE = re.compile(
    r'\\(?:cite|citep|citet|citealt|citeauthor|citeyear)\*?\{([^}]+)\}', re.M)
raw_calls = CITE_RE.findall(source)
cited_list = [k.strip() for call in raw_calls for k in call.split(",")]
cited_set  = set(cited_list)
print(f"Total cite calls : {len(raw_calls)}")
print(f"Unique cited keys: {len(cited_set)}")
for k in sorted(cited_set):
    print(f"  {k}")

Total cite calls : 29
Unique cited keys: 22
  chen2021evaluating
  cranmer2020lagrangian
  cranmer2023pysr
  hornik1989multilayer
  jin2020bayesian
  kambhampati2024llms
  koza1992gp
  koza1994genetic
  la_cava_2021_srbench
  landajuela2022unified
  lewkowycz2022solving
  lim2021temporal
  liu2022discover
  openai2023gpt4
  sahoo2018eql
  sun2022symbolic
  udrescu2020ai
  udrescu2020feynman
  uy2011semantically
  wei2022chain
  wolpert1997no
  xu2021neural


## Step 2 — Extract bibitem keys

In [3]:
BIBITEM_RE = re.compile(r'\\bibitem(?:\[[^\]]*\])?\{([^}]+)\}', re.M)
bibitem_keys = [k.strip() for k in BIBITEM_RE.findall(source)]
bibitem_set  = set(bibitem_keys)
print(f"Total bibitem entries : {len(bibitem_keys)}")
print(f"Unique bibitem keys   : {len(bibitem_set)}")
for k in sorted(bibitem_set):
    print(f"  {k}")

Total bibitem entries : 25
Unique bibitem keys   : 25
  chen2021evaluating
  cranmer2020lagrangian
  cranmer2023interp
  cranmer2023pysr
  hornik1989multilayer
  jin2020bayesian
  kambhampati2024llms
  karniadakis2021physics
  koza1992gp
  la_cava_2021_srbench
  landajuela2022unified
  lewkowycz2022solving
  lim2021temporal
  liu2022discover
  openai2023gpt4
  petersen2021deep
  raissi2019physics
  sahoo2018eql
  sun2022symbolic
  udrescu2020ai
  udrescu2020feynman
  uy2011semantically
  wei2022chain
  wolpert1997no
  xu2021neural


## Step 3 — Undefined keys and uncited entries

In [4]:
undefined = sorted(cited_set - bibitem_set)
uncited   = sorted(bibitem_set - cited_set)

print("=" * 60)
print(f"[CRITICAL] Undefined keys (cited but not in bib): {len(undefined)}")
print("=" * 60)
for k in undefined:
    lnums = [i+1 for i, ln in enumerate(lines) if k in ln and "bibitem" not in ln]
    print(f"  WARNING  {k!r:38s}  at lines {lnums[:6]}")

print()
print("=" * 60)
print(f"[INFO] Uncited bibitems: {len(uncited)}")
print("=" * 60)
for k in uncited:
    print(f"  INFO  {k!r}")

[CRITICAL] Undefined keys (cited but not in bib): 1
  WARNING  'koza1994genetic'                       at lines [327, 1888]

[INFO] Uncited bibitems: 4
  INFO  'cranmer2023interp'
  INFO  'karniadakis2021physics'
  INFO  'petersen2021deep'
  INFO  'raissi2019physics'


## Step 4 — Duplicate bibitem keys

In [5]:

counts = collections.Counter(bibitem_keys)
dupes  = {k: v for k, v in counts.items() if v > 1}
print("=" * 60)
print(f"[WARN] Duplicate bibitem keys: {len(dupes)}")
print("=" * 60)
for k, n in dupes.items():
    print(f"  DUPLICATE  {k!r}  defined {n}x")
if not dupes:
    print("  OK - no duplicates")

[WARN] Duplicate bibitem keys: 0
  OK - no duplicates


## Step 5 — Alias collisions (same paper, two keys)

In [6]:
ARXIV_RE = re.compile(r'arXiv[:\s]*(?:preprint\s+)?arXiv[:\s]*([\d]{4}\.\d+)', re.I)
bib_match = re.search(
    r'\\begin\{thebibliography\}(.*?)\\end\{thebibliography\}', source, re.DOTALL)
arxiv_map = collections.defaultdict(list)
if bib_match:
    for block in re.split(r'(?=\\bibitem)', bib_match.group(1)):
        km = BIBITEM_RE.match(block.strip())
        if not km:
            continue
        key = km.group(1).strip()
        for aid in ARXIV_RE.findall(block):
            arxiv_map[aid].append(key)
print("Alias collisions (same arXiv, different keys):")
found = False
for aid, keys in arxiv_map.items():
    if len(keys) > 1:
        print(f"  COLLISION  arXiv:{aid}  ->  {keys}")
        found = True
if not found:
    print("  OK - none found")

Alias collisions (same arXiv, different keys):
  COLLISION  arXiv:2305.01582  ->  ['cranmer2023pysr', 'cranmer2023interp']


## Step 6 — Fix recipe

In [7]:
recipe = (
    "FIX-B1  koza1994genetic is undefined.\n"
    "  Cited in section 3. Add bibitem OR change to koza1992gp.\n\n"
    "FIX-B2  cranmer2023pysr / cranmer2023interpretable are the same paper.\n"
    "  Remove cranmer2023interpretable; replace all uses with cranmer2023pysr.\n\n"
    "FIX-B3  udrescu2020ai / udrescu2020aifeynman are the same paper.\n"
    "  Remove udrescu2020aifeynman; replace all uses with udrescu2020ai.\n"
    "  One occurrence is in Definition 4.1.\n"
)
print(recipe)

FIX-B1  koza1994genetic is undefined.
  Cited in section 3. Add bibitem OR change to koza1992gp.

FIX-B2  cranmer2023pysr / cranmer2023interpretable are the same paper.
  Remove cranmer2023interpretable; replace all uses with cranmer2023pysr.

FIX-B3  udrescu2020ai / udrescu2020aifeynman are the same paper.
  Remove udrescu2020aifeynman; replace all uses with udrescu2020ai.
  One occurrence is in Definition 4.1.

